**Operation Anomaly Detection — STL Decomposition**

In [ ]:
from typing import Counter
from pandas.core.dtypes.missing import notna
import pandas as pd
import numpy as np
import re
import io
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
import ast

This project detects behavioral anomalies on a specific site at the operation level. Detection is driven by three complementary factors measured per hour:

- Operation Usage Count — total number of times the operation was invoked
- New IPs — distinct IP addresses appearing for the first time using the operation
- Active IPs — total number of unique IPs invoking the operation within the hour

In [ ]:
def z(f):
  g=(f-f.mean())/f.std()
  return g

with open("/content/demo_data12.csv",'r',encoding='utf-8') as f:
  content=f.read()

  df=pd.read_csv(io.StringIO(content))
  df['ip_list']=df['Column1'].apply(lambda x: x.split('|') if notna(x) else())
  df['ip_count']=df['ip_list'].apply(len)
  df = df.drop(columns=['Column1'])

  df['TimeGenerated [UTC]']=pd.to_datetime(df['TimeGenerated [UTC]'])
  df=df.set_index('TimeGenerated [UTC]')
  df=df.resample('h').sum().fillna(0)




plt.figure()
plt.plot(df['count_'])
plt.xlabel('Time')
plt.ylabel('Count')
plt.title('Hourly Count')
plt.show()

seen_IPs=set()
io=set()
for i in df['ip_list']:
  if not i:
      continue
  io.update(i)
  ioC=Counter(io)
  IP_df=pd.DataFrame(ioC,columns=['i','count'])
  IP_df2=(h for h in IP_df['i'] if IP_df['count']>1)
  seen_IPs.update(IP_df2)
  IP_rate=[]
  for i in df['ip_list']:
    if not i:
      IP_rate.append(0.0)
      continue
    IPf=[IP for IP in i if IP not in seen_IPs]
    IP_rate.append((len(IPf)/len(i)))

df['NewIPs_rate']=IP_rate

STL (Seasonal-Trend decomposition using LOESS) is applied independently to each factor, isolating the seasonal, trend, and residual components of the time series. The residual component — representing what the seasonal and trend models could not explain.

In [ ]:
stl=STL(df['count_'],period=24,seasonal=31,robust=True)
result=stl.fit()

stl2=STL(df['ip_count'],period=24,seasonal=31,robust=True)
result2=stl2.fit()

stl3=STL(df['NewIPs_rate'],period=24,seasonal=31,robust=True)
result3=stl3.fit()

fig=result.plot()
plt.show()

df['trend']=result.trend
df['seasonal']=result.seasonal
df['residual']=result.resid

df['trend2']=result2.trend
df['seasonal2']=result2.seasonal
df['residual2']=result2.resid

df['trend3']=result3.trend
df['seasonal3']=result3.seasonal
df['residual3']=result3.resid

df['count_z']=z(df['residual'])
df['IPs_z']=z(df['residual2'])
df['NewIPs_z']=z(df['residual3'])

The residual component is then used to compute a Z-score for each factor. Only positive residuals are considered anomalous, as negative deviations (lower-than-expected activity) fall outside the threat scope of this model. The three Z-scores are then combined into a weighted composite score, with each factor contributing a defined percentage to the final signal. Hours where the composite score exceeds the IQR-based upper bound are flagged as anomalies.

In [ ]:
df['anomaly_score']=np.where( (df['residual'] > 0),
(df['count_z']*0.7)+(df['IPs_z']*0.1)+(df['NewIPs_z']*0.2),0)
Q75=df['anomaly_score'].quantile(0.75)
Q25=df['anomaly_score'].quantile(0.25)
IQR=Q75-Q25
dynamic_threshold=Q75+(IQR*1.5)
print("dynamic_threshold")
print(dynamic_threshold)
max_reidual=df['residual'].max()
top_10=df
top_10=top_10.sort_values(by='residual',ascending=False)
print("top10")
print(top_10.head(10))
print("less110")
print(top_10.tail(10))
#top10
print("max1")
print(df[df['residual']==max_reidual])
final=df[df['anomaly_score']>dynamic_threshold]
final.to_excel("go.xlsx")

This layered approach ensures that a spike in a single factor alone is less likely to trigger a false positive — anomalies must manifest across the combined signal to be surfaced.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)


axes[0].plot(df.index, df['count_'],
             color='steelblue', linewidth=0.8, label='traffic')
axes[0].set_title('original traffic')
axes[0].set_ylabel('count')


axes[1].plot(df.index,df['anomaly_score'],
             color='gray', linewidth=0.8, label='residual')

axes[1].axhline(dynamic_threshold, color='red', linestyle='--',
                linewidth=1.5, label=f'95th percentile = {dynamic_threshold:.0f}')

axes[1].fill_between(df.index, dynamic_threshold, df['anomaly_score'],
                    where=(df['anomaly_score'] > dynamic_threshold),
                     color='red', alpha=0.4, label='anomaly zone')


axes[1].set_title('residual — red zone = anomaly')
axes[1].set_ylabel('residual')
axes[1].legend()

plt.tight_layout()
plt.show()